# CT107-3-3-TXSA — Group Assignment Part B
## Member 2 — XGBoost Sentiment Classification
**Dataset:** Sentiment140 (training.1600000.processed.noemoticon.csv)

---
**Instructions:** Place `training.1600000.processed.noemoticon.csv` in the same folder as this notebook before running.

## Step 0 — Install & Import Libraries

In [ ]:
# Install required libraries (run once)
# !pip install xgboost scikit-learn pandas nltk matplotlib seaborn

import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score,
                             precision_score, recall_score)
from xgboost import XGBClassifier

# Download NLTK data
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

print('All imports complete.')

---
## Step 1 — Load Raw Sentiment140 Dataset

In [ ]:
# Load raw Sentiment140 dataset
# Columns: target, ids, date, flag, user, text
col_names = ['target', 'ids', 'date', 'flag', 'user', 'text']

df = pd.read_csv(
    'training.1600000.processed.noemoticon.csv',
    encoding='latin-1',
    names=col_names
)

print(f'Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
print()
print('First 5 rows:')
print(df[['target', 'text']].head())

---
## Step 2 — Data Preparation & Pre-processing
*(Consistent with group's Section 1.5 pipeline)*

In [ ]:
# Remap target labels: 0 = Negative, 4 = Positive -> 0 and 1
df['target'] = df['target'].map({0: 0, 4: 1})

# Keep only relevant columns
df = df[['target', 'text']]

print('Label remapping complete.')
print(df['target'].value_counts())

In [ ]:
def clean_text(text):
    """
    Text normalization pipeline consistent with group's Section 1.5.
    1. Lowercase
    2. Remove URLs
    3. Remove user mentions (@username)
    4. Remove HTML entities
    5. Remove punctuation and digits
    6. Collapse whitespace
    7. Remove stop words and single characters
    """
    # Step 1: Lowercase
    text = text.lower()
    # Step 2: Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Step 3: Remove user mentions
    text = re.sub(r'@\w+', '', text)
    # Step 4: Remove HTML entities
    text = re.sub(r'&\w+;', '', text)
    # Step 5: Remove punctuation and digits
    text = re.sub(r'[^a-z\s]', '', text)
    # Step 6: Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Step 7: Remove stop words and single characters
    stop_words = set(stopwords.words('english'))
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 1]
    return ' '.join(tokens)

print('Cleaning pipeline defined. Applying to dataset...')
df['clean_text'] = df['text'].apply(clean_text)

# Drop empty rows resulting from aggressive cleaning
df = df[df['clean_text'].str.strip() != '']
df = df.dropna(subset=['clean_text'])

print(f'Cleaned dataset size: {df.shape[0]:,} rows')
print()
print('Sample cleaned text:')
print(df[['text', 'clean_text']].head(3).to_string())

---
## Step 3 — Balanced Sampling
*(100,000 rows — consistent with Logistic Regression member's approach)*

In [ ]:
# Sample 50,000 from each class for a balanced 100,000 row subset
neg_sample = df[df['target'] == 0].sample(n=50000, random_state=42)
pos_sample = df[df['target'] == 1].sample(n=50000, random_state=42)

df_sample = pd.concat([neg_sample, pos_sample]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Balanced sample size: {df_sample.shape[0]:,} rows')
print()
print('Class distribution in sample:')
print(df_sample['target'].value_counts())

---
## Step 4 — TF-IDF Feature Extraction

In [ ]:
# Split into features and labels
X = df_sample['clean_text']
y = df_sample['target']

# Train/test split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF Vectorization (max_features=5000 — consistent with group)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'Training set shape : {X_train_tfidf.shape}')
print(f'Testing set shape  : {X_test_tfidf.shape}')

---
# Section 2.1.2 — XGBoost Baseline Implementation

In [ ]:
print('=' * 60)
print('   SECTION 2.1.2 — XGBoost Baseline Model')
print('=' * 60)
print()

# Initialise XGBClassifier with baseline (default) parameters
xgb_baseline = XGBClassifier(
    n_estimators=100,        # Default: 100 boosting rounds
    max_depth=6,             # Default: maximum tree depth
    learning_rate=0.3,       # Default: step size shrinkage (eta)
    subsample=1.0,           # Default: fraction of samples per tree
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    tree_method='hist'       # Efficient histogram-based training
)

# Train the baseline model
print('Training XGBoost baseline model...')
xgb_baseline.fit(X_train_tfidf, y_train)
print('Training complete.')
print()

---
# Section 2.2.2 — XGBoost Evaluation Measures

In [ ]:
print('=' * 60)
print('   SECTION 2.2.2 — Evaluation Measures')
print('=' * 60)
print()
print('Evaluation metrics used for XGBoost:')
print()
print(f'{"Metric":<15} {"Formula":<50} {"Description"}')
print('-' * 100)
print(f'{"Accuracy":<15} {"(TP + TN) / (TP + TN + FP + FN)":<50} {"Proportion of all correct predictions"}')
print(f'{"Precision":<15} {"TP / (TP + FP)":<50} {"Of all predicted positives, how many were correct"}')
print(f'{"Recall":<15} {"TP / (TP + FN)":<50} {"Of all actual positives, how many were found"}')
print(f'{"F1-Score":<15} {"2 * (P * R) / (P + R)":<50} {"Harmonic mean of Precision and Recall"}')

---
# Section 2.3.2 — XGBoost Baseline Evaluation

In [ ]:
print('=' * 60)
print('   SECTION 2.3.2 — Baseline Model Evaluation')
print('=' * 60)
print()

# Generate predictions
y_pred_baseline = xgb_baseline.predict(X_test_tfidf)

# Calculate metrics
baseline_acc = accuracy_score(y_test, y_pred_baseline)
baseline_f1  = f1_score(y_test, y_pred_baseline, average='macro')
baseline_pre = precision_score(y_test, y_pred_baseline, average='macro')
baseline_rec = recall_score(y_test, y_pred_baseline, average='macro')

print(f'Accuracy  : {baseline_acc:.4f} ({baseline_acc*100:.2f}%)')
print(f'Precision : {baseline_pre:.4f}')
print(f'Recall    : {baseline_rec:.4f}')
print(f'F1-Score  : {baseline_f1:.4f}')
print()
print('Full Classification Report:')
print(classification_report(y_test, y_pred_baseline,
                            target_names=['Negative', 'Positive']))

In [ ]:
# Confusion Matrix — Baseline
cm_baseline = confusion_matrix(y_test, y_pred_baseline)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_baseline, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('XGBoost Baseline — Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('xgb_baseline_confusion_matrix.png', dpi=150)
plt.show()
print('Figure saved: xgb_baseline_confusion_matrix.png')

---
# Section 3.1.2 — XGBoost Hyperparameter Justification

In [ ]:
print('=' * 60)
print('   SECTION 3.1.2 — Hyperparameter Justification')
print('=' * 60)
print()
print('Hyperparameters selected for tuning:')
print()

hyperparams = {
    'n_estimators'  : '[100, 200, 300]      — Number of boosting trees (rounds)',
    'max_depth'     : '[3, 5, 6, 7]         — Maximum depth of each decision tree',
    'learning_rate' : '[0.01, 0.1, 0.2, 0.3] — Step size shrinkage (eta)',
    'subsample'     : '[0.6, 0.8, 1.0]      — Fraction of training samples per tree',
}

for param, desc in hyperparams.items():
    print(f'  {param:<20} : {desc}')

---
# Section 3.2.2 — Hyperparameter Tuning: RandomizedSearchCV

In [ ]:
print('=' * 60)
print('   SECTION 3.2.2 — RandomizedSearchCV Tuning')
print('=' * 60)
print()

# Define hyperparameter search space
param_dist = {
    'n_estimators'  : [100, 200, 300],
    'max_depth'     : [3, 5, 6, 7],
    'learning_rate' : [0.01, 0.1, 0.2, 0.3],
    'subsample'     : [0.6, 0.8, 1.0],
}

# Base estimator for tuning
xgb_tuning = XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42,
    tree_method='hist'
)

# RandomizedSearchCV — preferred over GridSearchCV for XGBoost
# because the larger hyperparameter space makes exhaustive search
# computationally prohibitive
random_search = RandomizedSearchCV(
    estimator=xgb_tuning,
    param_distributions=param_dist,
    n_iter=10,           # Test 10 random combinations
    cv=3,                # 3-fold cross-validation
    scoring='accuracy',
    verbose=2,
    random_state=42,
    n_jobs=-1            # Use all available CPU cores
)

print('Running RandomizedSearchCV (n_iter=10, cv=3)...')
print('This may take several minutes on large datasets.')
print()
random_search.fit(X_train_tfidf, y_train)

print()
print('Best Parameters Found:')
print(random_search.best_params_)
print()
print(f'Best Cross-Validation Accuracy: {random_search.best_score_:.4f}')

---
# Section 3.3.2 — Tuned Results & Performance Delta

In [ ]:
print('=' * 60)
print('   SECTION 3.3.2 — Tuned Model Results')
print('=' * 60)
print()

# Evaluate the best model found by RandomizedSearchCV
best_xgb = random_search.best_estimator_
y_pred_tuned = best_xgb.predict(X_test_tfidf)

# Calculate tuned metrics
tuned_acc = accuracy_score(y_test, y_pred_tuned)
tuned_f1  = f1_score(y_test, y_pred_tuned, average='macro')
tuned_pre = precision_score(y_test, y_pred_tuned, average='macro')
tuned_rec = recall_score(y_test, y_pred_tuned, average='macro')

print('Tuned Model — Classification Report:')
print(classification_report(y_test, y_pred_tuned,
                            target_names=['Negative', 'Positive']))

# Performance Delta Table
print()
print(f'{"Metric":<15} {"Baseline":<15} {"Tuned":<15} {"Delta"}')
print('-' * 60)
print(f'{"Accuracy":<15} {baseline_acc:<15.4f} {tuned_acc:<15.4f} {"+" if tuned_acc >= baseline_acc else ""}{(tuned_acc - baseline_acc):.4f}')
print(f'{"F1-Score":<15} {baseline_f1:<15.4f} {tuned_f1:<15.4f} {"+" if tuned_f1 >= baseline_f1 else ""}{(tuned_f1 - baseline_f1):.4f}')
print(f'{"Precision":<15} {baseline_pre:<15.4f} {tuned_pre:<15.4f} {"+" if tuned_pre >= baseline_pre else ""}{(tuned_pre - baseline_pre):.4f}')
print(f'{"Recall":<15} {baseline_rec:<15.4f} {tuned_rec:<15.4f} {"+" if tuned_rec >= baseline_rec else ""}{(tuned_rec - baseline_rec):.4f}')

In [ ]:
# Confusion Matrix — Tuned Model
cm_tuned = confusion_matrix(y_test, y_pred_tuned)

plt.figure(figsize=(7, 5))
sns.heatmap(cm_tuned, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('XGBoost Tuned — Confusion Matrix', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('xgb_tuned_confusion_matrix.png', dpi=150)
plt.show()
print('Figure saved: xgb_tuned_confusion_matrix.png')

In [ ]:
# Baseline vs Tuned — Side-by-side bar chart
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
baseline_vals = [baseline_acc, baseline_pre, baseline_rec, baseline_f1]
tuned_vals    = [tuned_acc,    tuned_pre,    tuned_rec,    tuned_f1]

x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, baseline_vals, width, label='Baseline', color='#4472C4')
bars2 = ax.bar(x + width/2, tuned_vals,    width, label='Tuned',    color='#ED7D31')

ax.set_title('XGBoost: Baseline vs Tuned Performance', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0.5, 1.0)
ax.set_ylabel('Score', fontsize=12)
ax.legend(fontsize=11)
ax.bar_label(bars1, fmt='%.4f', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.4f', padding=3, fontsize=9)
plt.tight_layout()
plt.savefig('xgb_baseline_vs_tuned.png', dpi=150)
plt.show()
print('Figure saved: xgb_baseline_vs_tuned.png')

---
## Summary

In [ ]:
print('=' * 60)
print('   XGBOOST — FINAL SUMMARY')
print('=' * 60)
print()
print(f'Best Parameters : {random_search.best_params_}')
print()
print(f'Baseline Accuracy : {baseline_acc:.4f} ({baseline_acc*100:.2f}%)')
print(f'Tuned Accuracy    : {tuned_acc:.4f} ({tuned_acc*100:.2f}%)')
print(f'Improvement       : +{(tuned_acc - baseline_acc)*100:.2f}%')
print()
print('Figures generated:')
print('  xgb_baseline_confusion_matrix.png')
print('  xgb_tuned_confusion_matrix.png')
print('  xgb_baseline_vs_tuned.png')